# Market regime detection with a Gaussian HMM

Runs the full pipeline from `src/` for one market. Set the ticker and exposure map in the settings cell, then run all. Tables are saved to `results/` and figures to `figures/`.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image
from scipy.optimize import brentq
from sklearn.metrics import roc_auc_score

from src.data import load_prices, emissions, clean_ticker, build_features, FEATURE_COLS
from src.hmm import select_n_states
from src.backtest import metrics, apply_costs, sweep_exposure_maps, prob_overlay
from src.crosscheck import fit_markov_switching, ranked_regime_path, agreement
from src.ticker_pipeline import hmm_pipeline, HEADLINE_MAP
from src.events import (vol_episodes, entry_exit_lags, false_alarms, fall_recovery,
                        threshold_states, alert_summary, spell_starts,
                        reliability, recalibrate)

# hmmlearn raises a numpy deprecation warning on every fit
warnings.filterwarnings('ignore', category=DeprecationWarning)
pd.options.display.float_format = '{:.3f}'.format

In [ ]:
TICKER = '^GSPC'        # any Yahoo Finance ticker, e.g. 'TLT', '^N225'
MAP_NAME = 'headline'   # 'headline' cuts exposure in crisis, 'mirror' raises it

# both maps fixed before any non-S&P run, so neither is tuned to a market's results
MAPS = {'headline': HEADLINE_MAP, 'mirror': {0: 0.0, 1: 0.5, 2: 1.0}}
EXPOSURE_MAP = MAPS[MAP_NAME]
NAME = clean_ticker(TICKER)

## 1. Data and features

In [ ]:
prices = load_prices(TICKER)
X, dates, next_day_ret = emissions(prices)
print(X.shape, dates.min().date(), '->', dates.max().date())

In [ ]:
# data checks: stale prices, impossible bars (0.01% slack for rounding in adjusted prices), extreme moves
close = prices['Close']
flat = close.diff() == 0
print('longest run of unchanged closes:', flat.groupby((~flat).cumsum()).sum().max())
print('closes outside High-Low:', int(((close > prices['High'] * 1.0001) | (close < prices['Low'] * 0.9999)).sum()))
close.pct_change().abs().nlargest(5)

## 2. Number of states

Fit on data before 2019, score BIC and log-likelihood on 2019 onward. On the S&P 500 both keep improving through 5 states, so there is no flat spot to pick; 3 is chosen because the gains decelerate and states beyond 3 only split the calm regime without a distinct interpretation. Every market uses the same 3 states so the results stay comparable. The 2019+ likelihood overlaps the walk-forward window, so this choice is not fully out of sample.

In [ ]:
train_end = int((dates < '2019-01-01').sum())
pd.DataFrame(select_n_states(X, train_end)).set_index('n')

## 3. Full-sample fit (interpretation only)

Viterbi path, state means, transition matrix, implied vs realised dwell times.

In [ ]:
# figures are saved for the headline map only, so a mirror run doesn't overwrite them
out = hmm_pipeline(TICKER, exposure_map=EXPOSURE_MAP, plots=MAP_NAME == 'headline')
states, results, ranked = out['states'], out['results'], out['ranked']
out['state_table']

In [ ]:
Image(f'figures/{NAME}_regime_price.png')

In [ ]:
Image(f'figures/{NAME}_transition_heatmap.png')

## 4. Walk-forward overlay vs baselines

Expanding window (`min_train=1000`, refit every 21 days), forward-filtered states. The fit takes about 6 minutes per market, so the states are cached in `data/` under the ticker and settings.

In [ ]:
print(states.index.min().date(), '->', states.index.max().date(), len(states), 'days')
out['regime_share']

In [ ]:
out['table']

In [ ]:
Image(f'figures/{NAME}_equity_curves.png')

In [ ]:
Image(f'figures/{NAME}_drawdowns.png')

### Probability-weighted exposure

Exposure is each state's map value weighted by its filtered probability, rather than the value for the most likely state, so it moves gradually when the model is unsure. The weights were fixed before the first run. Compared with the argmax overlay on the same days, gross and net of 5 bps.

In [ ]:
res_prob = prob_overlay(out['probs'], next_day_ret, EXPOSURE_MAP)
compare = {'argmax overlay': results['regime overlay'], 'probability-weighted': res_prob}
prob_table = pd.DataFrame({k: metrics(v['strat_ret'], v['exposure']) for k, v in compare.items()}).T
prob_table['sharpe_5bps'] = [metrics(apply_costs(v['strat_ret'], v['exposure'], 5), v['exposure'])['sharpe']
                             for v in compare.values()]
prob_table

## 5. Exposure maps and transaction costs

In [ ]:
maps = {'1.0 / 0.5 / 0.0 (headline)': HEADLINE_MAP,
        '1.0 / 0.6 / 0.2': {0: 1.0, 1: 0.6, 2: 0.2},
        '1.0 / 0.75 / 0.25': {0: 1.0, 1: 0.75, 2: 0.25},
        '1.0 / 1.0 / 0.0 (crisis only)': {0: 1.0, 1: 1.0, 2: 0.0}}
sweep_exposure_maps(states, next_day_ret, maps, cost_bps=5)

In [ ]:
# net Sharpe of every strategy at several per-trade cost assumptions
bps_grid = [0, 2, 5, 10]
net_sharpe = pd.DataFrame({
    name: [metrics(apply_costs(f['strat_ret'], f['exposure'], b), f['exposure'])['sharpe'] for b in bps_grid]
    for name, f in results.items()}, index=[f'{b} bps' for b in bps_grid])
net_sharpe

In [ ]:
# cost at which the overlay's net Sharpe falls to the vol-target's
def gap(bps):
    o, v = results['regime overlay'], results['vol target']
    return (metrics(apply_costs(o['strat_ret'], o['exposure'], bps), o['exposure'])['sharpe']
            - metrics(apply_costs(v['strat_ret'], v['exposure'], bps), v['exposure'])['sharpe'])

if gap(0) > 0 > gap(500):
    print(f'break-even cost: {brentq(gap, 0, 500):.1f} bps per unit of exposure traded')
else:
    print('no break-even in 0-500 bps: gap at 0 bps =', round(gap(0), 3), '| at 500 bps =', round(gap(500), 3))

## 6. Detection lag

How quickly the walk-forward model enters and leaves the crisis state, measured against a benchmark built from prices alone so it works on any ticker. A benchmark episode is 10 or more days on which the centred 11-day mean of the Parkinson range volatility is above its 90th percentile; its start and end are then set from the daily values within 5 days, and episodes 5 or fewer days apart are merged. Lags are model minus benchmark in trading days, so negative means early. Each model crisis spell is a real call (it overlaps an episode), an early warning (it ends up to 10 days before one) or a false alarm, and spells of 5 days or fewer are also marked as flips.

In [ ]:
bench = vol_episodes(prices)
lags = entry_exit_lags(states, bench)
lags

In [ ]:
fa = false_alarms(states, bench)
fa[fa['false_alarm'] | fa['early_warning']]

In [ ]:
lag_summary = alert_summary(states, bench)
lag_summary

### Falls vs recoveries

Each benchmark episode is split at its lowest close: the fall runs from the start to the trough (trough included), the recovery from the next day to the end. A model that tells the two apart should be in crisis through the fall but not the recovery.

In [ ]:
phases = fall_recovery(states, out['probs'], bench, prices['Close'])
phases

In [ ]:
# pooled over all phase days, so long episodes count for more
pooled = {}
for phase in ['fall', 'recov']:
    days = phases[f'{phase}_length']
    pooled[phase] = {
        'days': days.sum(),
        'crisis share': (phases[f'{phase}_crisis_share'] * days).sum() / days.sum(),
        'mean P(crisis)': (phases[f'{phase}_p'] * days).sum() / days.sum(),   # NaN rows have 0 days
    }
pd.DataFrame(pooled).T

### Rival: a volatility threshold

The simplest alert: crisis on days a vol measure is above the 90th percentile of its own past values (expanding window up to the previous day, 1000-day warm-up). Two rules, one per HMM input: Vol20 and the daily Parkinson range. The HMM earns its place only if it beats them on false alarms and flips at about the same entry lag. The benchmark is itself a 90th-percentile Parkinson rule (centred, with hindsight), so the Parkinson rule has a head start by construction.

In [ ]:
SPLIT = '2019-01-01'   # develop before this date; 2019 onward is the confirmation run

vol20 = build_features(prices)['Vol20']
# same daily Parkinson vol as vol_episodes
parkinson = np.sqrt(np.log(prices['High'] / prices['Low']) ** 2 / (4 * np.log(2)))

# rules cut to the HMM's walk-forward days, so all three are scored on the same days
detectors = {
    'HMM': states,
    'Vol20 rule': threshold_states(vol20).loc[states.index],
    'Parkinson rule': threshold_states(parkinson).loc[states.index],
}
pd.DataFrame({name: alert_summary(s[s.index < SPLIT], bench)
              for name, s in detectors.items()})

### Are flips single jump days?

Each crisis spell is described by its first day: `spike` (range over its previous 60-day median), `shock` (absolute return over the previous 60-day std) and `calm_before` (Vol20 the day before under its expanding 75th percentile). A day-t jump is spike >= 2 out of calm, known at the close. `next` (the following day's range on the same baseline) needs the next day, so it describes follow-through only and is never an input. Flips are jump days if at least half of flip false alarms are day-t jumps and that share is at least twice the share among real calls.

In [ ]:
starts = spell_starts(states[states.index < SPLIT], bench, prices)
starts.groupby('group').agg(spells=('length', 'size'),
                            day_t_jump=('day_t_jump', 'mean'),
                            jump_day=('jump_day', 'mean'),
                            spike=('spike', 'median'),
                            shock=('shock', 'median'),
                            calm_before=('calm_before', 'mean'),
                            next=('next', 'median'))

In [ ]:
# flips vs real calls only; AUC 0.5 = no separation, above 0.5 = flips score higher
pair = starts[starts['group'].isin(['flip', 'real'])]
is_flip = pair['group'] == 'flip'
auc = pd.Series({m: roc_auc_score(is_flip, pair[m]) for m in ['spike', 'shock', 'next']})

share_flip = pair.loc[is_flip, 'day_t_jump'].mean()
share_real = pair.loc[~is_flip, 'day_t_jump'].mean()
print(f'day-t jump share: flips {share_flip:.2f}, real calls {share_real:.2f}')
print('flips are jump days:', share_flip >= 0.5 and share_flip >= 2 * share_real)
auc

### Calibration of P(crisis)

When the model says P(crisis) = p, how often is a crisis really under way? Three outcomes: **daily** (a benchmark episode is under way that day; slow exits count against the top bins), **alert** (P(crisis) on the first day of each crisis spell vs whether the spell overlaps an episode) and **early warning** (days outside an episode vs one starting in the next 5 trading days). Brier skill compares with always forecasting the base rate (above 0 beats it). Calibrated if every bin with 30 or more days has |gap| <= 0.1.

In [ ]:
p_crisis = out['probs']['p2']
p_crisis = p_crisis[p_crisis.index < SPLIT]
in_episode = bench.loc[p_crisis.index] == 2
# first day of each episode; starts_soon looks at days t+1 to t+5 only
ep_start = (bench == 2) & (bench.shift(1) != 2)
starts_soon = pd.concat([ep_start.shift(-k, fill_value=False) for k in range(1, 6)],
                        axis=1).any(axis=1).loc[p_crisis.index]

outcomes = {
    'daily': (p_crisis, in_episode),
    'alert': (p_crisis.loc[starts.index], starts['overlaps']),   # starts from the jump-day cell
    'early warning': (p_crisis[~in_episode], starts_soon[~in_episode]),
}
tables = {name: reliability(p, y) for name, (p, y) in outcomes.items()}
pd.concat(tables, names=['outcome'])

In [ ]:
scores = {}
for name, (p, y) in outcomes.items():
    y = y.astype(float)
    brier = ((p - y) ** 2).mean()
    brier_base = ((y.mean() - y) ** 2).mean()          # always forecasting the base rate
    big = tables[name][tables[name]['days'] >= 30]
    scores[name] = {'n': len(y), 'base rate': y.mean(), 'brier': brier,
                    'brier skill': 1 - brier / brier_base,
                    # no bin with 30+ days (alert) means the rule can't be checked
                    'calibrated': (big['gap'].abs() <= 0.1).all() if len(big) else None}
pd.DataFrame(scores).T

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], color='grey', lw=1, ls='--', label='perfect')
for name, t in tables.items():
    t = t[t['days'] > 0]
    ax.plot(t['mean_p'], t['observed'], marker='o', label=name)
ax.set(xlabel='mean P(crisis) in bin', ylabel='observed share',
       title=f'{NAME}: calibration before {SPLIT[:4]}')
ax.legend()
plt.show()

### Recalibration

An isotonic map from raw to calibrated P(crisis), learned on days before `SPLIT` and applied from `SPLIT` onward. It only goes up, so alert days and entry lags do not change; only what the number means does. Kept if it lowers the Brier score after `SPLIT`. The base rate forecast uses the share of episode days before `SPLIT`, the only rate known then.

In [ ]:
# daily outcome on the full walk-forward; the map never sees days from SPLIT onward
p_all = out['probs']['p2']
y_all = bench.loc[p_all.index] == 2
fit_days = p_all.index < SPLIT
p_test, y_test = p_all[~fit_days], y_all[~fit_days]
p_cal = recalibrate(p_all[fit_days], y_all[fit_days], p_test)

base = y_all[fit_days].mean()
brier = pd.Series({'raw': ((p_test - y_test) ** 2).mean(),
                   'calibrated': ((p_cal - y_test) ** 2).mean(),
                   'base rate': ((base - y_test) ** 2).mean()})
brier['calibrated skill'] = 1 - brier['calibrated'] / brier['base rate']
brier

In [ ]:
pd.concat({'raw': reliability(p_test, y_test),
           'calibrated': reliability(p_cal, y_test)}, names=['p'])

## 7. Cross-check: Markov-switching regression

`MarkovRegression` on the 100x log return only (switching mean and variance), against the HMM's full-sample Viterbi path.

In [ ]:
ms = fit_markov_switching(pd.Series(X[:, FEATURE_COLS.index('Return')], index=dates), k_regimes=3)
sigma2 = np.array([ms.params[f'sigma2[{i}]'] for i in range(3)])
const = np.array([ms.params[f'const[{i}]'] for i in range(3)])
o = np.argsort(sigma2)
pd.DataFrame({'mean_ret': const[o], 'vol': np.sqrt(sigma2[o]), 'expected_dwell': np.asarray(ms.expected_durations)[o]},
             index=['calm', 'choppy', 'crisis'])

In [ ]:
ms_path = ranked_regime_path(ms, dates)
ag = agreement(pd.Series(ranked, index=dates), ms_path)
print(f"same-day agreement: {ag['same_day_pct']:.1f}%  |  crisis-day Jaccard: {ag['crisis_jaccard']:.2f}")
ag['confusion']

## 8. Save results

One CSV per market and map in `results/`: the metrics table plus Sharpe net of 5 bps. The detection-lag episode table is saved per market. Written findings are in the README.

In [ ]:
RESULTS_DIR = Path('results')
RESULTS_DIR.mkdir(exist_ok=True)
summary = out['table'].copy()
summary['sharpe_5bps'] = net_sharpe.loc['5 bps']
summary['map'] = MAP_NAME
summary.to_csv(RESULTS_DIR / f'{NAME}_{MAP_NAME}.csv', index_label='strategy', float_format='%.4f')
lags.to_csv(RESULTS_DIR / f'{NAME}_lags.csv', index=False)
summary